# 2026.10.13(화) 20h30~

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense, Dropout, Activation

# 학습데이터 생성
# ------------------------------------------------------------------
# train_data = np.random.randint(0, 10, size=(10000, 5))
#  - 자료형: NumPy의 ndarray(다차원 배열). 파이썬 리스트가 아니다.
#      · 모양(shape) = (10000, 5)  → 2차원, 10000행 x 5열
#        (행 하나 = 학습 샘플 하나 = 숫자 5개짜리 "문장")
#      · 원소 자료형(dtype) = int64 (정수. 환경에 따라 int32일 수 있음)
#  - np.random.randint(0, 10, ...) : 0 이상 10 "미만"의 정수, 즉 0~9 중에서
#      무작위로 뽑는다. (끝값 10은 포함되지 않음)
#  - size=(10000, 5) : 결과 배열의 모양. 5개짜리 숫자열을 10000개 만든다.
#  - 예) [[3 7 0 2 9]
#         [1 1 4 8 5]
#         ... (10000줄)]
#  - 확인하려면: type(train_data) → numpy.ndarray, train_data.shape → (10000, 5),
#               train_data.dtype → int64
# ------------------------------------------------------------------
train_data = np.random.randint(0, 10, size=(10000, 5))
# ------------------------------------------------------------------
# label_data = np.flip(train_data, axis=1)  : 정답(레이블) 만들기
#  - np.flip(배열, axis=1) : 열 방향(axis=1, 가로)으로 순서를 뒤집는다.
#      [3 7 0 2 9] → [9 2 0 7 3]  (각 행 안에서 5개 숫자를 역순으로)
#      axis=0이면 행 방향(세로)으로 뒤집혀 10000개 행의 순서가 바뀌므로 여기서는 쓰지 않는다.
#  - 그래서 이 모델의 문제는 "숫자 5개를 받아 역순으로 출력하기"다.
#  - label_data도 shape (10000, 5)의 ndarray. 입력과 같은 길이라서
#    모델 출력 (샘플 수, 5, 10)의 각 위치가 0~9 중 무엇인지(분류)를 맞히는 문제가 된다.
#  - 참고: np.flip은 원본을 복사하지 않고 "뒤집힌 보기(view)"를 줄 수 있다.
#    여기서는 train_data를 나중에 바꾸지 않으므로 문제없다.
# ------------------------------------------------------------------
label_data = np.flip(train_data, axis=1)
# 앞의 4개 샘플만 찍어서 입력(속성)과 정답(클래스)이 정말 역순인지 눈으로 확인
print(train_data[:4])
print(label_data[:4])

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
#pad_sequence 전처리
# pad_sequences : 길이가 다른 시퀀스를 같은 길이(maxlen)로 맞춘다.
#  - 짧으면 앞쪽을 0으로 채우고(padding='pre'), 길면 앞쪽을 잘라낸다.
#  - 여기서는 모든 샘플이 이미 길이 5라서 maxlen=5로 해도 값이 그대로다(형식만 맞춰 주는 단계).
#  - 단, 0은 패딩 값이기도 하고 이 데이터에서는 실제 숫자 0이기도 하다. 이 실습에서는
#    패딩이 실제로 일어나지 않으니 문제가 없지만, 길이가 다른 데이터에서는 헷갈릴 수 있다.
padded_train = pad_sequences(train_data, maxlen=5)
# 첫 번째 샘플 하나만 출력 (shape (5,)의 1차원 배열)
print(padded_train[0])

2026-10-13 21:01:17.747951: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[[0 8 1 3 9]
 [3 4 0 9 9]
 [9 8 0 9 9]
 [7 6 8 8 5]]
[[9 3 1 8 0]
 [9 9 0 4 3]
 [9 9 0 8 9]
 [5 8 8 6 7]]
[0 8 1 3 9]


In [3]:
# 모델링
# Sequential : 층을 위에서 아래로 차례로 쌓는 가장 단순한 모델 틀
model = Sequential()
# ------------------------------------------------------------------
# [층 1] Embedding(10, 100, input_length=5)
#  - 역할: 숫자(단어) 번호 하나를 100개짜리 실수 벡터로 바꾼다. (번호 → 의미 벡터)
#  - 10  : 어휘 크기. 입력에 나올 수 있는 값이 0~9 열 가지라서 10이다.
#          (입력에 10 이상이 들어오면 범위를 벗어나 오류가 난다)
#  - 100 : 임베딩 차원. 숫자 하나를 길이 100짜리 벡터로 표현한다.
#          정답이 있는 값이 아니라 직접 정하는 하이퍼파라미터다.
#          (숫자가 10종류뿐이라 100은 넉넉한 값이다. 더 작게 해도 학습될 수 있다.)
#  - input_length=5 : 입력 시퀀스의 길이(숫자 5개). train_data의 열 수와 같다.
#          (Keras 3에서는 deprecated라 경고가 나올 수 있고, 없어도 동작한다.)
#  - 입력 shape : (샘플 수, 5)        정수 5개
#    출력 shape : (샘플 수, 5, 100)   숫자마다 100차원 벡터가 붙어 3차원이 된다
#  - 학습 파라미터 수 : 10 x 100 = 1,000
#    벡터는 처음엔 랜덤이고 학습하면서 문제에 도움이 되도록 조금씩 바뀐다.
# ------------------------------------------------------------------
model.add(Embedding(10,100,input_length=5))
# ------------------------------------------------------------------
# [층 2] Bidirectional(LSTM(64, return_sequences=True))
#  - LSTM(64) : 순서가 있는 데이터를 앞에서부터 한 칸씩 읽으며 "기억"을 쌓는 순환 층.
#      · 64 = 은닉 상태(기억)의 크기. 각 위치마다 길이 64짜리 벡터를 출력한다.
#      · 한 칸을 읽을 때마다 직전까지의 기억과 현재 입력을 합쳐 기억을 갱신한다.
#  - return_sequences=True : 마지막 위치의 결과만 주는 게 아니라 5개 위치 전부의 결과를 준다.
#      · False(기본값)이면 출력 (샘플 수, 64), True이면 (샘플 수, 5, 64).
#      · 이 문제는 위치마다 정답(역순 숫자)이 있으므로 5개 위치 모두의 출력이 필요하다.
#  - Bidirectional : LSTM을 두 개 만들어 하나는 왼쪽→오른쪽, 다른 하나는 오른쪽→왼쪽으로
#      읽게 하고 두 결과를 이어 붙인다(기본 merge_mode='concat').
#      · 역순 출력에서는 첫 출력 자리에 "맨 뒤 숫자"가 와야 한다. 한 방향 LSTM은 앞쪽
#        위치에서 뒤쪽 숫자를 볼 수 없어서, 양방향이 이 문제에 유리하다.
#      · 출력 shape : (샘플 수, 5, 64 x 2 = 128)
#  - 학습 파라미터 수 : LSTM 하나 = 4 x (64 x (100 + 64) + 64) = 42,240
#      (4 = 내부 게이트 4개, 100 = 입력 차원, 64 = 은닉 크기, 마지막 64 = 편향)
#      양방향이라 x 2 = 84,480
# ------------------------------------------------------------------
model.add(Bidirectional(LSTM(64, return_sequences=True)))
# ------------------------------------------------------------------
# [층 3] Dense(10, activation='softmax')
#  - 역할: 각 위치(5곳)마다 "0~9 중 어떤 숫자인가"를 확률로 내놓는 분류 층.
#  - 10 : 출력 뉴런 수 = 분류할 클래스 수(숫자 0~9).
#  - softmax : 10개 출력을 합이 1인 확률로 바꾼다. 가장 높은 값이 예측 숫자다.
#  - 3차원 입력 (샘플 수, 5, 128)에 Dense를 쓰면 마지막 축(128)에만 적용되어
#    5개 위치 각각에 같은 가중치가 쓰인다. 출력 shape : (샘플 수, 5, 10)
#  - 학습 파라미터 수 : 128 x 10 + 10(편향) = 1,290
#  - 전체 파라미터 : 1,000 + 84,480 + 1,290 = 86,770 (model.summary()의 Total params와 비교)
# ------------------------------------------------------------------
model.add(Dense(10,activation='softmax')) # 분류기

# 모델의 요약을 보기 위해 한 번 모델을 호출하여 초기화합니다
# (None, 5) : 샘플 수는 아직 정해지지 않았음(None), 길이는 5
model.build(input_shape=(None, 5))
model.summary()

/opt/anaconda3/envs/voice-ai/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 100)         │         1,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 5, 128)         │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5, 10)          │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,770 (338.95 KB)

 Trainable params: 86,770 (338.95 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
#모델 컴파일
# loss='sparse_categorical_crossentropy' : 정답이 one-hot이 아니라 정수(0~9)일 때 쓰는 다중 분류 손실.
#   정답 label_data (10000, 5)의 정수와 모델 출력 (샘플 수, 5, 10)의 확률을 위치별로 비교한다.
#   (정답을 one-hot으로 바꿨다면 'categorical_crossentropy'를 쓴다)
# optimizer='adam' : 학습률을 알아서 조절해 주는 널리 쓰는 최적화 방법
# metrics=['accuracy'] : 학습 중 정확도도 같이 출력 (위치별 정확도의 평균)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',metrics=['accuracy'])
#모델 실행
history = model.fit(padded_train, label_data, epochs=10, batch_size=32, validation_split=0.2)
# loss와 accuracy
loss, acc = model.evaluate(padded_train, label_data)
print(f"loss : {loss:.4f}, acc = {acc:.4f}")

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.4481 - loss: 1.4768 - val_accuracy: 0.5401 - val_loss: 1.0701
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.5636 - loss: 0.9847 - val_accuracy: 0.5903 - val_loss: 0.9069
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 28ms/step - accuracy: 0.6141 - loss: 0.8563 - val_accuracy: 0.6381 - val_loss: 0.8180
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.6770 - loss: 0.7440 - val_accuracy: 0.7174 - val_loss: 0.6812
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7522 - loss: 0.6256 - val_accuracy: 0.7773 - val_loss: 0.5886
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.8265 - loss: 0.4987 - val_accuracy: 0.8474 - val_loss: 0.4495
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.8829 - loss: 0.3735 - val_accuracy: 0.9039 - val_loss: 0.3286
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.9255 - loss: 0.2666 - val

In [8]:
# 입력받아 예측하는 프로그램 작성
while True:
    user = input("0~9 사이의 숫자 5개 또는 Quit: ")
    if user == 'Quit':
        print("종료합니다")
        break
    numbers = list(map(int, user.split()))
    array_numbers = np.array([numbers])
    padded_numbers = pad_sequences(array_numbers, maxlen=5)
    print(f"padded_numbers = {padded_numbers}")
    prediction = model.predict(padded_numbers)
    print(np.argmax(prediction, axis=-1)[0])

padded_numbers = [[1 3 5 7 9]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
[9 7 5 3 1]
종료합니다
